# 🧪 Chapter 8: Pytest for DevOps

📖 Book: Chapter 8, pp. 307–343 · ⏱️ 2–3 hours · 🧱 Needs: Ch 3, 7 · Level: Intermediate

---

> **🎯 Key idea:** `pytest` replaces `unittest` boilerplate with bare `assert`s, `tmp_path`, `parametrize`, and `testinfra` live host validation.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** `tmpdir` (`py.path.local`)  
> **🆕 Now (2026):** `tmp_path` (`pathlib.Path`)  
> **Why:** Modern stdlib `Path` API

> **📜 Then (book, ~2020):** `nose` comparisons in book  
> **🆕 Now (2026):** `pytest` exclusively (`testinfra` & `nbval` maintained)  
> **Why:** `nose` is unmaintained/dead

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
You refactored a port-parsing helper on Friday afternoon, and Monday morning production deploys fail because `'80'` vs `80` broke. Manual testing doesn't scale across edge cases or server configurations. With `pytest` and `testinfra`, both your Python logic and live host state are verified automatically.

### Concept: `pytest` Fixtures & `parametrize`
**What:** Dependency-injected setup & data-driven tests  
**Why:** Keeps tests DRY with rich AST assertion diffs  
**How:** `@pytest.fixture` + `@pytest.mark.parametrize`  
**When:** All unit & integration tests

📖 *Book listing:* `TestStrToBool` Parametrize & `monkeypatch` (pp. 314–325)

**Plain-language analogy:** `pytest` fixtures are like a clean lab bench prepared fresh before each experiment and wiped down automatically afterward.

**Walkthrough:** The book replaces repetitive `TestCase` methods with `@pytest.mark.parametrize` and uses `monkeypatch.setattr` to mock `subprocess.check_output` safely without touching disks.

### Concept: Infrastructure Testing (`testinfra`)
**What:** `pytest` plugin for live hosts  
**Why:** Verifies packages, `systemd` services & open ports  
**How:** Uses `host` fixture over SSH/Docker/Ansible  
**When:** Post-provision smoke validation

📖 *Book listing:* `testinfra` Nginx Host Validation (pp. 326–337)

**Plain-language analogy:** `testinfra` is like a building inspector walking into a newly built server and checking that the plumbing (`nginx`), power (`systemd`), and doors (port `80`) actually work.

**Walkthrough:** The book uses `testinfra`'s `host` fixture over SSH/Ansible to assert that `nginx` is installed, its `systemd` service is running, and TCP port `80` is listening.

## 6. 🟢 Beginner Examples (Real `pytest` Execution!)

#### Example 1: Beginner Example 1: Write & run a real pytest file using pytest.main()
*Expected output:* Simulated fallback if optional package is missing.

In [1]:
# Beginner Example 1: Write & run a real pytest file using pytest.main()
import tempfile
from pathlib import Path

TEST_CODE_1 = '''
def string_to_bool(val: str) -> bool:
    return val.strip().lower() in {'yes', '1', 'true'}

def test_truthy():
    assert string_to_bool('Yes') is True

def test_falsy():
    assert string_to_bool('no') is False
'''
try:
    import pytest
    with tempfile.TemporaryDirectory() as tmp:
        tf = Path(tmp) / 'test_basic.py'
        tf.write_text(TEST_CODE_1, encoding='utf-8')
        rc = pytest.main([str(tf), '-v'])
        assert rc == 0
except ImportError:
    print('pytest not installed; install via: pip install pytest')

============================= test session starts ==============================
platform linux -- Python 3.13.14, pytest-9.2.0, pluggy-1.7.0 -- /home/vannaboth/.cache/uv/builds-v0/.tmpcAe6Ra/bin/python
cachedir: .pytest_cache
rootdir: /tmp/tmp6oaq1f3r
plugins: platformdirs-4.14.8, anyio-4.16.0
collecting ... 

collected 2 items

../../../../../../tmp/tmp6oaq1f3r/test_basic.py::test_truthy 

PASSED      [ 50%]


../../../../../../tmp/tmp6oaq1f3r/test_basic.py::test_falsy 

PASSED       [100%]



============================== 2 passed in 0.01s ===============================


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

#### Example 2: Beginner Example 2: Parametrize + pytest.raises in real pytest
*Expected output:* Simulated fallback if optional package is missing.

In [2]:
# Beginner Example 2: Parametrize + pytest.raises in real pytest
TEST_CODE_2 = '''
import pytest

def validate_port(p: int) -> int:
    if not (1 <= p <= 65535):
        raise ValueError(f'Bad port: {p}')
    return p

@pytest.mark.parametrize('port', [22, 80, 443, 8080])
def test_valid_ports(port):
    assert validate_port(port) == port

@pytest.mark.parametrize('bad', [0, -1, 70000])
def test_invalid_ports(bad):
    with pytest.raises(ValueError):
        validate_port(bad)
'''
try:
    import pytest
    with tempfile.TemporaryDirectory() as tmp:
        tf = Path(tmp) / 'test_ports.py'
        tf.write_text(TEST_CODE_2, encoding='utf-8')
        assert pytest.main([str(tf), '-q']) == 0
except ImportError:
    print('pytest not installed')

.

.

.

.

.

.

.

                                                                  [100%]


7 passed in 0.01s


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

#### Example 3: Beginner Example 3: Built-in monkeypatch & tmp_path fixtures
*Expected output:* Simulated fallback if optional package is missing.

In [3]:
# Beginner Example 3: Built-in monkeypatch & tmp_path fixtures
TEST_CODE_3 = '''
import os

def test_env_and_tmp(monkeypatch, tmp_path):
    monkeypatch.setenv('DEPLOY_ENV', 'staging')
    cfg = tmp_path / 'env.txt'
    cfg.write_text(os.environ['DEPLOY_ENV'], encoding='utf-8')
    assert cfg.read_text(encoding='utf-8') == 'staging'
'''
try:
    import pytest
    with tempfile.TemporaryDirectory() as tmp:
        tf = Path(tmp) / 'test_fixtures.py'
        tf.write_text(TEST_CODE_3, encoding='utf-8')
        assert pytest.main([str(tf), '-q']) == 0
except ImportError:
    print('pytest not installed')

.

                                                                        [100%]


1 passed in 0.00s


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 7. 🟡 Intermediate Examples: `conftest.py` Factory Fixture & `testinfra`

#### Example 4: Intermediate Example 1: conftest.py shared fixture + test module
*Expected output:* Simulated fallback if optional package is missing.

In [4]:
# Intermediate Example 1: conftest.py shared fixture + test module
CONFTEST_CODE = '''
import pytest
@pytest.fixture
def sample_inventory(tmp_path):
    p = tmp_path / 'hosts.txt'
    p.write_text('web-01\\nweb-02\\n', encoding='utf-8')
    return p
'''
TEST_INV = '''
def test_inventory_count(sample_inventory):
    hosts = sample_inventory.read_text(encoding='utf-8').splitlines()
    assert len(hosts) == 2
'''
try:
    import pytest
    with tempfile.TemporaryDirectory() as tmp:
        (Path(tmp) / 'conftest.py').write_text(CONFTEST_CODE, encoding='utf-8')
        (Path(tmp) / 'test_inv.py').write_text(TEST_INV, encoding='utf-8')
        assert pytest.main([tmp, '-q']) == 0
except ImportError:
    print('pytest not installed')

.

                                                                        [100%]


1 passed in 0.01s


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): Testinfra host validation
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): Testinfra host validation
# Requires: pip install testinfra
try:
    import testinfra
    print('testinfra installed:', testinfra.__version__)
except ImportError:
    from types import SimpleNamespace
    mock_host = SimpleNamespace(
        package=lambda n: SimpleNamespace(is_installed=(n == 'nginx')),
        service=lambda n: SimpleNamespace(is_running=(n == 'nginx')),
    )
    assert mock_host.package('nginx').is_installed and mock_host.service('nginx').is_running
    print('testinfra not installed; simulated host check PASSED!')

testinfra not installed; simulated host check PASSED!


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Pre-Deploy Service Config Validator** tested via `unittest` (stdlib) so CI gates work even without plugins.

#### Example 6: import json, tempfile
*Expected output:* Printed labelled output below.

In [6]:
import json, tempfile
from pathlib import Path

def validate_service_config(cfg_path: Path) -> list[str]:
    if not cfg_path.exists():
        return ['Missing config']
    data = json.loads(cfg_path.read_text(encoding='utf-8'))
    errs = []
    if not isinstance(data.get('port'), int):
        errs.append('port must be int')
    if not str(data.get('health', '')).startswith('/'):
        errs.append('health must start with /')
    return errs

with tempfile.TemporaryDirectory() as tmp:
    p = Path(tmp) / 'svc.json'
    p.write_text(json.dumps({'port': 8080, 'health': '/health'}))
    assert validate_service_config(p) == []
    print('Config validator PASSED!')

Config validator PASSED!


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Real Tool Demo (`shutil.which` + `pytest`)
*Expected output:* `1 passed` from real `pytest` (or graceful skip).

In [7]:
# Real Tool Demo: run pytest.main() on a temp test file (guarded with shutil.which)
import shutil, subprocess, tempfile
from pathlib import Path

if shutil.which('pytest') or shutil.which('python3'):
    with tempfile.TemporaryDirectory() as tmp:
        tf = Path(tmp) / 'test_smoke.py'
        tf.write_text('def test_health_ok():\n    assert 200 == 200\n', encoding='utf-8')
        cp = subprocess.run(['python3', '-m', 'pytest', str(tf), '-q'], capture_output=True, text=True)
        if cp.returncode == 0:
            print('Real pytest run:', cp.stdout.strip())
        else:
            print('⏭️ pytest not installed in active python3, skipping')

Real pytest run: .                                                                        [100%]
1 passed in 0.00s


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Patching wrong module path** -> Mock never intercepts call. ✅ Patch where symbol is looked up.
- ❌ **Mutating `os.environ` directly** -> Leaks state to later tests. ✅ Use `monkeypatch.setenv()`.
- ❌ **Missing `test_` prefix** -> `pytest` skips the test silently. ✅ Name files `test_*.py` & funcs `test_*`.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement is_valid_port and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [8]:
# Starter Exercise Cell (implement is_valid_port and run to verify!)
def _run_starter_checks():
    assert is_valid_port(80) and not is_valid_port(True)
def is_valid_port(port: int) -> bool:
    raise NotImplementedError('Implement is_valid_port')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement is_valid_port


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** How does `pytest` discover tests?

<details>
<summary>✅ Answer 1</summary>

Scans `test_*.py` / `*_test.py` for `test_*` functions and `Test*` classes.

</details>

**Q2:** What is `conftest.py` for?

<details>
<summary>✅ Answer 2</summary>

Auto-loaded fixture/hook sharing across a test directory.

</details>

**Q3:** Why prefer `tmp_path` over `tmpdir` in modern `pytest`?

<details>
<summary>✅ Answer 3</summary>

`tmp_path` returns a standard `pathlib.Path` object instead of legacy `py.path.local`.

</details>

**Q4:** How does `monkeypatch` prevent test pollution?

<details>
<summary>✅ Answer 4</summary>

Automatically reverts patched env vars/attributes after each test.

</details>

**Q5:** What backends does `testinfra` support?

<details>
<summary>✅ Answer 5</summary>

`local`, `ssh`, `paramiko`, `docker`, `ansible`, `kubernetes`, `salt`.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Ex 1**: Check `not isinstance(port, bool)` because `isinstance(True, int)` is `True` in Python!

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [9]:
# Complete Solutions + Assert Verification
import unittest
from types import SimpleNamespace
from unittest.mock import patch

def is_valid_port(port: int) -> bool:
    return isinstance(port, int) and not isinstance(port, bool) and 1 <= port <= 65535

http_client = SimpleNamespace(fetch_status=lambda url: 200)

def check_remote_health(url: str) -> bool:
    try:
        return http_client.fetch_status(url) == 200
    except TimeoutError:
        return False

assert is_valid_port(80) and not is_valid_port(True) and not is_valid_port(70000)
with patch.object(http_client, 'fetch_status', side_effect=[200, 503, TimeoutError('slow')]):
    assert check_remote_health('http://svc') is True
    assert check_remote_health('http://svc') is False
    assert check_remote_health('http://svc') is False
print('All Chapter 8 assertions PASSED!')

All Chapter 8 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `pytest.main([path, '-v'])` — Run pytest from Python (e.g. `assert pytest.main([tmp]) == 0`)
- `with pytest.raises(ValueError):` — Assert expected exception (e.g. `pytest.raises(KeyError)`)
- `pytest --nbval nb.ipynb` — Validate notebooks in CI (e.g. Checks outputs match)

### ✅ What You Can Now Do
- Write `pytest` tests using bare `assert`, `tmp_path`, `@pytest.mark.parametrize`, and `pytest.raises`.
- Isolate external commands and env vars with `monkeypatch` and `unittest.mock.patch`.
- Run `pytest.main()` programmatically and understand `testinfra` live-host checks.

### ➡️ Next Chapter
Chapter 9 builds on this by applying Python concurrency, backoff retries, and `boto3` patterns in the cloud.